## Conversion of text files to Conllu format with Stanza
## Processors (here GRC - Ancient Greek) can be defined individually; directory structure is mirrored

In [8]:
import stanza
import os
import time
import unicodedata
import sys
from datetime import datetime
from stanza.utils.conll import CoNLL

def initialize_pipeline(language, processors, model_dir):
    """
    Initializes the Stanza pipeline with the PROIEL-UD model for Ancient Greek.
    """
    processor_list = ",".join(processors.keys())
    
    stanza.download(lang=language, processors=processor_list, model_dir=model_dir)

    return stanza.Pipeline(
        lang=language,
        processors=processors,
        use_gpu=True,
        model_dir=model_dir
    )

def process_text_file(nlp, input_path, conllu_path, language, processors):
    """
    Processes a text file with the NLP pipeline and saves the results in CoNLL-U format.
    """
    with open(input_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    conllu_text = ""
    
    # Add metadata as a comment at the beginning of the file
    metadata = (
        f"# Language: {language}\n"
        f"# Processors: {processors}\n"
        f"# Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"# Stanza Version: {stanza.__version__}\n"
        f"# Python Version: {sys.version.split()[0]}\n"
    )
    conllu_text += metadata + "\n"
    
    for line in lines:
        line = line.strip()
        if line:
            doc = nlp(line)
            
            for sentence in doc.sentences:
                for word in sentence.words:
                    conllu_text += (
                        f"{word.id}\t{word.text}\t{word.lemma}\t{word.upos}\t"
                        f"{word.xpos}\t{word.feats}\t{word.head}\t{word.deprel}\t"
                        f"{word.deps}\t{word.misc}\n"
                    )
                conllu_text += "\n"
        else:
            conllu_text += "\n"
    
    with open(conllu_path, 'w', encoding='utf-8') as file:
        file.write(conllu_text)

def process_directory(input_dir, output_dir, language, processors, model_dir):
    """
    Iterates through the directory and processes all text and CoNLL-U files.
    """
    nlp = initialize_pipeline(language, processors, model_dir)
    
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".txt"):
                input_path = os.path.join(root, file)
                relative_path = os.path.relpath(input_path, input_dir)
                conllu_path = os.path.join(output_dir, "conllu", relative_path.replace(".txt", ".conllu"))
                
                os.makedirs(os.path.dirname(conllu_path), exist_ok=True)
                
                start_time = time.time()
                process_text_file(nlp, input_path, conllu_path, language, processors)
                end_time = time.time()
                print(f"File '{input_path}' processed in {end_time - start_time:.2f} seconds.")

# Main execution
if __name__ == "__main__":
    input_directory = "./test-grc"
    output_directory = "./test-grc-out"
    language = "grc"
    model_directory = "F:\\Python-Res\\stanza_resources"

    # Individuelle Modelle für jeden Prozessor-Typ
    processors = {
        "tokenize": "perseus",  # Standard-Tokenizer
        "pos": "perseus_nocharlm",  # Standard POS-Tagger
        "lemma": "perseus_nocharlm",  # Spezielles Lemma-Modell (Perseus)
        "depparse": "perseus_nocharlm"  # Standard Dependency Parser
    }

    process_directory(input_directory, output_directory, language, processors, model_directory)


2025-02-12 09:35:09 INFO: Downloaded file to F:\Python-Res\stanza_resources\resources.json
2025-02-12 09:35:09 INFO: Downloading these customized packages for language: grc (Ancient_Greek)...
| Processor | Package          |
--------------------------------
| tokenize  | perseus          |
| pos       | perseus_nocharlm |
| lemma     | perseus_nocharlm |
| depparse  | perseus_nocharlm |
| pretrain  | conll17          |

2025-02-12 09:35:09 INFO: File exists: F:\Python-Res\stanza_resources\grc\tokenize\perseus.pt
2025-02-12 09:35:09 INFO: File exists: F:\Python-Res\stanza_resources\grc\pos\perseus_nocharlm.pt
2025-02-12 09:35:09 INFO: File exists: F:\Python-Res\stanza_resources\grc\lemma\perseus_nocharlm.pt
2025-02-12 09:35:10 INFO: File exists: F:\Python-Res\stanza_resources\grc\depparse\perseus_nocharlm.pt
2025-02-12 09:35:10 INFO: File exists: F:\Python-Res\stanza_resources\grc\pretrain\conll17.pt
2025-02-12 09:35:10 INFO: Finished downloading models and saved to F:\Python-Res\stanza

2025-02-12 09:35:10 INFO: Downloaded file to F:\Python-Res\stanza_resources\resources.json
2025-02-12 09:35:10 INFO: Loading these models for language: grc (Ancient_Greek):
| Processor | Package          |
--------------------------------
| tokenize  | perseus          |
| pos       | perseus_nocharlm |
| lemma     | perseus_nocharlm |
| depparse  | perseus_nocharlm |

2025-02-12 09:35:10 WARNING: GPU requested, but is not available!
2025-02-12 09:35:10 INFO: Using device: cpu
2025-02-12 09:35:10 INFO: Loading: tokenize
2025-02-12 09:35:10 INFO: Loading: pos
2025-02-12 09:35:11 INFO: Loading: lemma
2025-02-12 09:35:12 INFO: Loading: depparse
2025-02-12 09:35:12 INFO: Done loading processors!


File './test-grc\txt\chapter06.txt' processed in 3.16 seconds.
File './test-grc\txt\chapter07.txt' processed in 4.23 seconds.
